# Build the 6-category dataset (small sample per category)

This notebook takes a **small, equal-sized sample** from one public source per category and writes one file, `dataset.csv`, with fixed train / val / test splits. Everyone trains on that same file, so results are comparable.

| Category | Source | How it's loaded | Notes |
|---|---|---|---|
| Request | Enron emails, reviewed in notebook 01 | `review_done.csv` you upload | real emails |
| Correspondence | Enron emails, reviewed in notebook 01 | same file | real emails |
| Complaint | CFPB consumer complaint narratives | Kaggle dataset you add as input | real complaints |
| Application | Resume dataset (24 job fields) | Hugging Face `Divyaamith/Kaggle-Resume` | resumes stand in for job applications |
| Contract | CUAD commercial contracts | zip from Zenodo (~106 MB) | first 3,000 characters of each contract |
| Invoice | Invoice OCR text | Hugging Face `mychen76/invoices-and-receipts_ocr_v1` | OCR of synthetic invoices |
| Request (optional top-up) | Bitext customer-support messages | Hugging Face, **training split only** | off by default (see Step 6) |

**Before you run (Kaggle):**
1. *Settings → Internet on* (GPU not needed).
2. *Add Input* → search **"us-consumer-finance-complaints"** (publisher CFPB) and add it. Any CFPB complaints CSV with a narrative column works.
3. *Add Input → Upload* your reviewed Enron file **`review_done.csv`** (or upload it to the working folder).

## Step 0: Settings

`N_PER_CLASS` is the sample size per category. 300 is enough to start; raise it to 500 later for the M3 target. Every source is sampled with the same seed, so re-running gives the same dataset.

In [ ]:
import ast, glob, json, os, re, zipfile
import numpy as np
import pandas as pd
import requests
from datasets import load_dataset

N_PER_CLASS = 300
SEED = 42
MIN_CHARS, MAX_CHARS = 80, 3000   # drop near-empty docs; cut very long ones
USE_BITEXT = False                 # see Step 6
INPUT_DIR = "/kaggle/input"
WORK_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."

parts = []  # one DataFrame per source: text, label, source, train_only

def add(texts, label, source, train_only=False, min_chars=MIN_CHARS, limit=None):
    df = pd.DataFrame({"text": list(texts)})
    df["text"] = df["text"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip().str.slice(0, MAX_CHARS)
    df = df[df["text"].str.len() >= min_chars].drop_duplicates("text")
    if limit is not None:
        df = df.head(limit)
    df["label"], df["source"], df["train_only"] = label, source, train_only
    parts.append(df)
    print(f"{label:<15} {source:<22} {len(df):>5} rows")

def find_file(name):
    hits = glob.glob(name) + glob.glob(os.path.join(WORK_DIR, name)) + glob.glob(f"{INPUT_DIR}/**/{name}", recursive=True)
    if not hits:
        raise FileNotFoundError(f"{name} not found: upload it to the notebook (Add Input or the working folder)")
    return hits[0]

## Step 1: Request and Correspondence (reviewed Enron emails)

These come from notebook 01. We keep the subject line because it carries a lot of the meaning ("Request for...", "FYI: ..."). If a class has more than `N_PER_CLASS` rows, we sample down.

In [ ]:
enron = pd.read_csv(find_file("review_done.csv"))
enron = enron[enron["final_label"].isin(["Request", "Correspondence"])]
enron["full"] = enron["subject"].fillna("").astype(str) + ". " + enron["text"].astype(str)
for label in ["Request", "Correspondence"]:
    rows = enron[enron["final_label"] == label]
    add(rows.sample(n=min(N_PER_CLASS, len(rows)), random_state=SEED)["full"], label, "enron")

## Step 2: Complaint (CFPB narratives)

The notebook looks through your Kaggle inputs for a CSV with a column whose name contains "narrative", so it works with any CFPB complaints export. It reads only that column and only the first 300,000 rows, to save memory.

The CFPB replaces personal details with `XXXX`. That marker would let a model recognize "this came from CFPB" instead of "this is a complaint", so the repo's `clean_text` removes it before training.

In [ ]:
def find_narrative_csv():
    for path in glob.glob(f"{INPUT_DIR}/**/*.csv", recursive=True):
        cols = pd.read_csv(path, nrows=0).columns
        match = [c for c in cols if "narrative" in c.lower()]
        if match:
            return path, match[0]
    raise FileNotFoundError("No CSV with a 'narrative' column in /kaggle/input. Add the CFPB complaints dataset as input.")

path, col = find_narrative_csv()
print("using", path, "| column:", col)
cfpb = pd.read_csv(path, usecols=[col], nrows=300_000, low_memory=False)[col].dropna()
cfpb = cfpb[cfpb.str.len().between(MIN_CHARS, 4000)]
add(cfpb.sample(n=min(N_PER_CLASS, len(cfpb)), random_state=SEED), "Complaint", "cfpb")

## Step 3: Application (resumes)

No public dataset of real job or permit applications exists, so resumes stand in for them. We sample evenly across the 24 job fields so one field doesn't dominate. Say this openly in the data card: it's a known limitation for the error analysis.

In [ ]:
resumes = load_dataset("Divyaamith/Kaggle-Resume", split="train").to_pandas()
text_col = "Resume_str" if "Resume_str" in resumes.columns else [c for c in resumes.columns if "str" in c.lower() or "text" in c.lower()][0]
cat_col = "Category" if "Category" in resumes.columns else None
resumes = resumes.dropna(subset=[text_col])
if cat_col:
    per_cat = int(np.ceil(N_PER_CLASS / resumes[cat_col].nunique()))
    resumes = resumes.groupby(cat_col, group_keys=False).apply(lambda g: g.sample(n=min(per_cat, len(g)), random_state=SEED))
add(resumes.sample(n=min(N_PER_CLASS, len(resumes)), random_state=SEED)[text_col], "Application", "resumes")

## Step 4: Contract (CUAD)

CUAD's 510 commercial contracts come as plain-text files inside a zip on Zenodo (about 106 MB, a minute to download). We keep the first 3,000 characters of each, which holds the title, the parties and the opening clauses. That's what a person routing the document would look at anyway.

In [ ]:
zip_path = os.path.join(WORK_DIR, "CUAD_v1.zip")
if not os.path.exists(zip_path):
    url = "https://zenodo.org/records/4595826/files/CUAD_v1.zip?download=1"
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(zip_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)

with zipfile.ZipFile(zip_path) as z:
    txt_files = [n for n in z.namelist()
                 if n.lower().endswith(".txt") and "txt" in os.path.dirname(n).lower() and "__macosx" not in n.lower()]
    print("contract text files found:", len(txt_files))
    rng = np.random.default_rng(SEED)
    chosen = rng.choice(txt_files, size=min(N_PER_CLASS, len(txt_files)), replace=False)
    contracts = [z.read(n).decode("utf-8", errors="ignore")[:MAX_CHARS] for n in chosen]
add(contracts, "Contract", "cuad")

## Step 5: Invoice (OCR text)

Each row's `raw_data` holds the OCR output as a list of words in reading order. We join the words into one text. Streaming means only the rows we need are downloaded, not all 280 MB.

In [ ]:
def ocr_text(raw):
    data = raw
    if isinstance(raw, str):
        for parse in (json.loads, ast.literal_eval):
            try:
                data = parse(raw)
                break
            except Exception:
                continue
    words = data.get("ocr_words") if isinstance(data, dict) else None
    if isinstance(words, str):
        try:
            words = ast.literal_eval(words)
        except Exception:
            words = None
    if isinstance(words, list):
        return " ".join(str(w) for w in words)
    return " ".join(re.findall(r"'([^']*)'", str(raw)))  # fallback: every quoted fragment

stream = load_dataset("mychen76/invoices-and-receipts_ocr_v1", split="train", streaming=True)
try:
    stream = stream.remove_columns(["image"])
except Exception:
    pass
invoices = []
for row in stream:
    invoices.append(ocr_text(row["raw_data"]))
    if len(invoices) >= 2 * N_PER_CLASS:
        break
invoices = pd.Series(invoices).sample(frac=1, random_state=SEED).head(N_PER_CLASS)
add(invoices, "Invoice", "invoice_ocr")

## Step 6 (optional): Top up Request with Bitext

Real Enron Requests are limited. Bitext has thousands of customer requests, but they are short chat messages with placeholders like `{{Order Number}}`, a strong source marker. If you turn this on, Bitext rows go **into the training split only**, never val/test, so the evaluation stays on real documents. A cleaner fix is another review round in notebook 01 (Step 9).

In [ ]:
if USE_BITEXT:
    have = sum(len(p) for p in parts if (p["label"] == "Request").all())
    need = max(0, N_PER_CLASS - have)
    if need:
        bitext = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset", split="train").to_pandas()
        # Bitext messages are short chat lines, so they get a lower length minimum
        pool = bitext["instruction"].sample(frac=1, random_state=SEED)
        add(pool, "Request", "bitext", train_only=True, min_chars=30, limit=need)
else:
    print("Bitext top-up is off")

## Step 7: Combine and split

- Drop any text that appears twice (across sources too).
- Split each category **70 / 15 / 15** into train / val / test, stratified, with a fixed seed.
- `train_only` rows (Bitext) always go to train.

The `source` column stays in the file, so the error analysis can check whether mistakes cluster by source (risk R5, source leakage).

In [ ]:
data = pd.concat(parts, ignore_index=True).drop_duplicates("text").reset_index(drop=True)
rng = np.random.default_rng(SEED)
data["split"] = "train"
for label, group in data[~data["train_only"]].groupby("label"):
    idx = rng.permutation(group.index.to_numpy())
    n_test, n_val = round(0.15 * len(idx)), round(0.15 * len(idx))
    data.loc[idx[:n_test], "split"] = "test"
    data.loc[idx[n_test:n_test + n_val], "split"] = "val"

data = data[["text", "label", "source", "split"]]
out_path = os.path.join(WORK_DIR, "dataset.csv")
data.to_csv(out_path, index=False)
print("saved", out_path, data.shape)
print(pd.crosstab(data["label"], data["split"], margins=True))
print(pd.crosstab(data["label"], data["source"]))
print("median characters per category:")
print(data["text"].str.len().groupby(data["label"]).median().astype(int))

## Step 8: Look at a few rows before using them

Read two random documents per category. If something looks wrong (an empty invoice, a contract that is only a table of contents), fix it now: everything downstream trusts this file.

In [ ]:
for label, group in data.groupby("label"):
    print(f"===== {label} =====")
    for t in group.sample(n=min(2, len(group)), random_state=1)["text"]:
        print("-", t[:300].replace("\n", " "), "...\n")

## Next

Download `dataset.csv` (Output panel) and put it in the repo at `data/processed/dataset.csv`. Don't commit it to git: the `.gitignore` already blocks it. Share it with the team through Kaggle (*New Dataset* from this output) or a shared drive. Then:

```
python -m docrouter.train_baseline --model logreg
```

**Known differences between sources** (for the data card): document length varies a lot between categories (contracts and resumes are long, emails short). If the baseline scores near 100%, suspect it is learning length or source style, and check the per-source errors.